## Course data paths

Run the setup cell below before the lesson. Bundled inputs are resolved from the course repository, so paths also work after moving the notebook into `code/`. Generated practice files still use the notebook working directory.

This lesson needs an instructor-supplied CSV under the module's `datasets/external/` folder. See the module dataset README before running the file-loading section.


In [ ]:
from pathlib import Path

# Find the cloned course repository from the notebook or repository folder.
COURSE_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "docs" / "course-catalog.json").is_file()),
    None,
)
if COURSE_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned course repository.")


# 🤖 Machine Learning from Scratch
## Complete Linear Regression Curriculum

---

> **A concept-first, math-first, intuition-first course.**  
> We build complete understanding before every line of code.

### 📚 What You Will Learn
| Phase | Topic |
|-------|-------|
| 1 | Foundations of Machine Learning |
| 2 | Linear Regression — the core model |
| 3 | Cost Function (MSE) — deep dive |
| 4 | Derivatives — the math engine |
| 5 | Gradient Descent — the learning algorithm |
| 6 | Training Process — step by step |
| 7 | Visualization — seeing learning happen |
| 8 | Full Implementation from scratch |

**Prerequisites:** Basic Python, basic algebra (no calculus required — we derive everything intuitively)

---
# 🔵 PHASE 1: Foundations of Machine Learning
---

## 1.1 What is Machine Learning?

**Traditional Programming:**
```
Rules + Data → Output
```
You write explicit rules. The computer follows them exactly.

**Machine Learning:**
```
Data + Output → Rules (learned automatically)
```
You provide examples. The algorithm discovers the rules itself.

---

### 💡 Analogy
> Teaching a child to recognize dogs. You don't recite rules:  
> *"four legs + fur + barks = dog"*  
> You show them 500 pictures. Their brain learns the pattern.  
> **That is machine learning** — learning patterns from examples.

---

## 1.2 What is a Model?

A **model** is a mathematical function:  
$$f(X) = \hat{y}$$

- **Input:** Features $X$ (what you know)
- **Output:** Prediction $\hat{y}$ (what you want to know)
- **Parameters:** Internal values (weights) that are *tuned* during training

Before training → random predictions (terrible)  
After training → accurate predictions (good)

---

## 1.3 Features (X) and Targets (y)

| Term | Symbol | Meaning | Example |
|------|--------|---------|--------|
| Features | $X$ | Input variables you provide | house size, bedrooms |
| Target | $y$ | What you want to predict | sale price |
| Prediction | $\hat{y}$ | Model's output (y-hat) | predicted price |
| Error | $\hat{y} - y$ | How wrong the prediction is | -$5,000 |

---

## 1.4 Real-World Example: House Price Prediction

**Given:** A dataset of houses where we know size and price.  
**Goal:** Given a NEW house size → predict its price.

```
Size (sq ft)  →  [MODEL]  →  Predicted Price ($)
    1,400     →  f(1400)  →  $183,000
```

> ⚠️ **Common Misconception:** ML models are NOT thinking like humans.  
> They are mathematical functions whose parameters are tuned to minimize prediction error.  
> No magic. Just optimization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dataset
X = np.array([650, 785, 900, 1050, 1200, 1350, 1500, 1650, 1800, 2000])
y = np.array([70, 95, 110, 130, 155, 172, 195, 215, 235, 260])

print("📊 Our Training Dataset")
print("=" * 40)
print(f"{'House':>6}  {'Size (sq ft)':>12}  {'Price ($k)':>10}")
print("-" * 40)
for i, (xi, yi) in enumerate(zip(X, y)):
    print(f"{'House '+str(i+1):>6}  {xi:>12,.0f}  {yi:>10,.0f}")
print("=" * 40)
print(f"\nDataset summary:")
print(f"  Samples      : {len(X)}")
print(f"  Size range   : {X.min():,.0f} – {X.max():,.0f} sq ft")
print(f"  Price range  : ${y.min():,.0f}k – ${y.max():,.0f}k")
print(f"  Avg price    : ${y.mean():,.0f}k")
# Plot
plt.scatter(X, y)

# Labels
plt.xlabel("House Size (sq ft)")
plt.ylabel("Price ($k)")
plt.title("House Price vs Size")

# Show plot
plt.show()

In [ ]:
# ── Plot 1: Visualize the raw data ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(X, y, s=100, color='#5b8af5', edgecolors='#8fb4ff',
           linewidth=1, zorder=5, label='Training houses')

# Annotate each point
for xi, yi in zip(X, y):
    ax.annotate(f'${yi:.0f}k', (xi, yi),
                xytext=(8, 4), textcoords='offset points',
                fontsize=8, color='#9ba3bc')

ax.set_xlabel('House Size (sq ft)', fontsize=12)
ax.set_ylabel('Price ($k)', fontsize=12)
ax.set_title('Phase 1 — Raw Data: House Size vs. Price\n'
             'Goal: find the line that best fits this data', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

# Add question annotation
ax.annotate('← What price?', xy=(1400, 185), fontsize=11,
            color='#f7a94b', style='italic',
            arrowprops=dict(arrowstyle='->', color='#f7a94b'))
ax.axvline(x=1400, color='#f7a94b', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.show()
print("\n💡 The question: Given a NEW house of 1,400 sq ft, what price should we predict?")

---
# 🟢 PHASE 2: Linear Regression — The Core Model
---

## 2.1 The Model Equation

Linear Regression fits a **straight line** through data:

$$\hat{y} = m \cdot x + b$$

| Symbol | Name | Role |
|--------|------|------|
| $\hat{y}$ | Predicted value | Output of the model |
| $x$ | Input feature | House size (sq ft) |
| $m$ | Slope / weight | How much price changes per sq ft |
| $b$ | Intercept / bias | Base price when size = 0 |

---

## 2.2 Intuition: Slope (m)

**Slope** answers: *"For every 1 sq ft increase in size, how much does price change?"*

- If $m = 0.15$: every extra sq ft adds $150 to the price
- Larger $m$ → steeper line (price grows faster with size)
- Negative $m$ → inverse relationship (bigger = cheaper — unlikely here!)

## 2.3 Intuition: Intercept (b)

**Intercept** is the predicted price when $x = 0$.

- In our housing context: it's the "base cost" (land, taxes, etc.)
- $b$ shifts the entire line up or down without rotating it
- A house of size 0 isn't real — $b$ is a mathematical convenience

---

## 2.4 The Problem

There are **infinitely many lines** we could draw. Which is best?

We need a way to measure "how good" a line is → **Cost Function** (Phase 3)

> ⚠️ **Common Misconception:** "Linear" means the relationship between  
> features and output is linear — NOT that the data must look like a line.  
> With multiple features, it's a flat hyperplane in higher dimensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Prediction function
def predict(X, m, b):
    return m * X + b

# Different lines (guesses)
lines = [
    (0.05, 50, "Line A"),
    (0.12, 20, "Line B"),
    (0.10, 80, "Line C"),
]

# Range for drawing lines
x_range = np.linspace(500, 2100, 300)

# Plot
plt.figure(figsize=(8, 5))

# Actual data
plt.scatter(X, y, label="Actual Data")

# Plot each line
for m, b, name in lines:
    y_line = predict(x_range, m, b)
    plt.plot(x_range, y_line, linestyle='--', label=f"{name} (m={m}, b={b})")

# Labels
plt.xlabel("House Size (sq ft)")
plt.ylabel("Price ($k)")
plt.title("Which Line Fits Best? (ŷ = m·x + b)")
plt.legend()
plt.grid()

plt.show()

# Predictions for 1400 sq ft
print("\nPredictions for 1400 sq ft:")
for m, b, name in lines:
    pred = predict(1400, m, b)
    print(f"{name}: {pred:.1f}k")

print("\nActual nearby price: ~183k")

---
# 🟡 PHASE 3: The Cost Function — Mean Squared Error (MSE)
---

## 3.1 What is a Cost Function?

A **cost function** measures how wrong the model currently is — in a single number.  
Lower cost = better model. Our goal: **minimize the cost.**

### 💡 Analogy: The Archer
> An archer fires 10 arrows at a target. The cost function = total distance of all arrows from the bullseye.  
> Training = adjusting the archer's aim to minimize total distance.

---

## 3.2 Why Not Just Average the Errors?

Error for one sample: $e_i = \hat{y}_i - y_i$

**Problem with simple average:**  
If error = +100 for house 1 and error = −100 for house 2:  
Average = (100 + (−100)) / 2 = **0** ← looks perfect! But you're wildly wrong on both.

**Solution:** Square each error before summing.

---

## 3.3 Mean Squared Error (MSE)

$$\text{MSE}(m, b) = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2 = \frac{1}{n} \sum_{i=1}^{n} (m \cdot x_i + b - y_i)^2$$

---

## 3.4 Three Reasons We Square the Error

| Reason | Explanation |
|--------|-------------|
| **1. Eliminates negatives** | $(−50)^2 = 2500 = (50)^2$ — both same magnitude |
| **2. Penalizes large errors** | Error of 10 → 100. Error of 20 → 400. Four times more! Non-linear penalty. |
| **3. Differentiable** | $x^2$ has clean derivative $2x$. Absolute value $|x|$ is not differentiable at 0. |

---

> ⚠️ **Common Confusion:** Why divide by $n$?  
> To compare models across different dataset sizes. 1000 samples always has higher raw sum than 10 — dividing by $n$ normalizes to a per-sample average error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Data
X = np.array([1, 2, 3, 4, 5])
y = np.array([3, 5, 7, 9, 11])

# Step 2: Calculate m (slope)
# m = (y2 - y1) / (x2 - x1)
m = (y[1] - y[0]) / (X[1] - X[0])

# Step 3: Calculate b (intercept)
# b = y - mx
b = y[0] - m * X[0]

print(f"Slope (m) = {m}")
print(f"Intercept (b) = {b}")
print(f"Equation: y = {m}x + {b}")

# Step 4: Predictions using the equation
y_pred = m * X + b

# Step 5: Visualization
plt.figure()

# Actual points
plt.scatter(X, y)

# Line (model)
plt.plot(X, y_pred)

plt.title("Simple Linear Regression Example")
plt.xlabel("X")
plt.ylabel("y")

plt.show()

In [ ]:
# ── Simplified MSE step-by-step ─────────────────────────────

def compute_mse(y_pred, y_true):
    errors = y_pred - y_true
    squared_errors = errors ** 2
    return np.mean(squared_errors)

# Test line
m_test, b_test = 0.05, 50
y_pred_test = predict(X, m_test, b_test)

print("\n📊 Step-by-Step MSE Calculation")
print(f"Model: ŷ = {m_test}·x + {b_test}")
print("-" * 65)
print(f"{'i':>3} {'x':>6} {'y(actual)':>12} {'ŷ(pred)':>10} {'err':>8} {'err²':>10}")
print("-" * 65)

total_sq_error = 0

for i, (xi, yi, yp) in enumerate(zip(X, y, y_pred_test), start=1):
    err = yp - yi
    err_sq = err ** 2
    total_sq_error += err_sq

    print(f"{i:>3} {xi:>6.0f} {yi:>12.1f} {yp:>10.1f} {err:>8.1f} {err_sq:>10.1f}")


# Final MSE
mse = total_sq_error / len(X)

print("-" * 65)
print(f"{'Sum of squared errors':>50} = {total_sq_error:>10.1f}")
print(f"{'MSE':>50} = {mse:>10.2f}")

print(f"\n✗ MSE = {mse:.2f} → High = Bad fit")

# Plot actual vs predicted + errors
plt.figure(figsize=(8,5))

# Scatter actual data
plt.scatter(X, y, color='blue', label='Actual')

# Plot the predicted line
plt.plot(X, y_pred_test, color='red', linestyle='--', label=f'Predicted Line (m={m_test},b={b_test})')

# Draw vertical lines showing errors
for xi, yi, yp in zip(X, y, y_pred_test):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1)

plt.xlabel("House Size (sq ft)")
plt.ylabel("Price ($k)")
plt.title("Actual vs Predicted (Errors as vertical lines)")
plt.legend()
plt.grid(True)
plt.show()

### Gradeint Descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sample data (assuming same as your code)
X = np.array([1000, 1500, 2000, 2500, 3000])
y = np.array([200, 250, 300, 350, 400])

def predict(X, m, b):
    return m * X + b

def compute_mse(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)

# Learning rate
lr = 0.0000001  # Small lr for stability

# Initial parameters
m, b = 0.05, 50
n = len(X)






In [ ]:
# ─────────────────────────────────────────────────────────────
# ITERATION 0 (Initial State)
# ─────────────────────────────────────────────────────────────
print(f"\n🔵 ITERATION 0 (Initial)")
print(f"m = {m:.6f}, b = {b:.2f}")

y_pred = predict(X, m, b)
mse = compute_mse(y_pred, y)
print(mse)

In [ ]:


# Gradients: ∂MSE/∂m and ∂MSE/∂b
dm = (2/n) * np.sum((y_pred - y) * X)  # Gradient w.r.t m
db = (2/n) * np.sum(y_pred - y)        # Gradient w.r.t b

print(f"MSE = {mse:.2f}")
print(f"Gradients: dm = {dm:.2f}, db = {db:.2f}")

plt.figure(figsize=(7,4))
plt.scatter(X, y, color='blue', s=60, label='Actual', zorder=3)
plt.plot(X, y_pred, 'r--', label=f'Predicted (m={m:.4f}, b={b:.1f})', linewidth=2)
for xi, yi, yp in zip(X, y, y_pred):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1.5)
plt.title(f"Iteration 0 | MSE={mse:.2f}")
plt.xlabel("Size")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# ─────────────────────────────────────────────────────────────
# ITERATION 1
# ─────────────────────────────────────────────────────────────
m = m - lr * dm  # Update m
b = b - lr * db  # Update b

print(f"\n🔵 ITERATION 1")
print(f"m = {m:.6f}, b = {b:.2f}")

y_pred = predict(X, m, b)
mse = compute_mse(y_pred, y)
dm = (2/n) * np.sum((y_pred - y) * X)
db = (2/n) * np.sum(y_pred - y)

print(f"MSE = {mse:.2f}")
print(f"Gradients: dm = {dm:.2f}, db = {db:.2f}")

plt.figure(figsize=(7,4))
plt.scatter(X, y, color='blue', s=60, label='Actual', zorder=3)
plt.plot(X, y_pred, 'r--', label=f'Predicted (m={m:.4f}, b={b:.1f})', linewidth=2)
for xi, yi, yp in zip(X, y, y_pred):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1.5)
plt.title(f"Iteration 1 | MSE={mse:.2f}")
plt.xlabel("Size")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(mse)

In [ ]:
# ─────────────────────────────────────────────────────────────
# ITERATION 2
# ─────────────────────────────────────────────────────────────
m = m - lr * dm
b = b - lr * db

print(f"\n🔵 ITERATION 2")
print(f"m = {m:.6f}, b = {b:.2f}")

y_pred = predict(X, m, b)
mse = compute_mse(y_pred, y)
dm = (2/n) * np.sum((y_pred - y) * X)
db = (2/n) * np.sum(y_pred - y)

print(f"MSE = {mse:.2f}")
print(f"Gradients: dm = {dm:.2f}, db = {db:.2f}")

plt.figure(figsize=(7,4))
plt.scatter(X, y, color='blue', s=60, label='Actual', zorder=3)
plt.plot(X, y_pred, 'r--', label=f'Predicted (m={m:.4f}, b={b:.1f})', linewidth=2)
for xi, yi, yp in zip(X, y, y_pred):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1.5)
plt.title(f"Iteration 2 | MSE={mse:.2f}")
plt.xlabel("Size")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ─────────────────────────────────────────────────────────────
# ITERATION 3
# ─────────────────────────────────────────────────────────────
m = m - lr * dm
b = b - lr * db

print(f"\n🔵 ITERATION 3")
print(f"m = {m:.6f}, b = {b:.2f}")

y_pred = predict(X, m, b)
mse = compute_mse(y_pred, y)
dm = (2/n) * np.sum((y_pred - y) * X)
db = (2/n) * np.sum(y_pred - y)

print(f"MSE = {mse:.2f}")
print(f"Gradients: dm = {dm:.2f}, db = {db:.2f}")

plt.figure(figsize=(7,4))
plt.scatter(X, y, color='blue', s=60, label='Actual', zorder=3)
plt.plot(X, y_pred, 'r--', label=f'Predicted (m={m:.4f}, b={b:.1f})', linewidth=2)
for xi, yi, yp in zip(X, y, y_pred):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1.5)
plt.title(f"Iteration 3 | MSE={mse:.2f}")
plt.xlabel("Size")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ITERATION 4
# ─────────────────────────────────────────────────────────────
m = m - lr * dm
b = b - lr * db

print(f"\n🔵 ITERATION 4")
print(f"m = {m:.6f}, b = {b:.2f}")

y_pred = predict(X, m, b)
mse = compute_mse(y_pred, y)
dm = (2/n) * np.sum((y_pred - y) * X)
db = (2/n) * np.sum(y_pred - y)

print(f"MSE = {mse:.2f}")
print(f"Gradients: dm = {dm:.2f}, db = {db:.2f}")

plt.figure(figsize=(7,4))
plt.scatter(X, y, color='blue', s=60, label='Actual', zorder=3)
plt.plot(X, y_pred, 'r--', label=f'Predicted (m={m:.4f}, b={b:.1f})', linewidth=2)
for xi, yi, yp in zip(X, y, y_pred):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1.5)
plt.title(f"Iteration 4 | MSE={mse:.2f}")
plt.xlabel("Size")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ─────────────────────────────────────────────────────────────
# ITERATION 5
# ─────────────────────────────────────────────────────────────
m = m - lr * dm
b = b - lr * db

print(f"\n🔵 ITERATION 5")
print(f"m = {m:.6f}, b = {b:.2f}")

y_pred = predict(X, m, b)
mse = compute_mse(y_pred, y)

print(f"MSE = {mse:.2f}")
print(f"\n✅ Final Model: ŷ = {m:.6f}·x + {b:.2f}")

plt.figure(figsize=(7,4))
plt.scatter(X, y, color='blue', s=60, label='Actual', zorder=3)
plt.plot(X, y_pred, 'r--', label=f'Predicted (m={m:.4f}, b={b:.1f})', linewidth=2)
for xi, yi, yp in zip(X, y, y_pred):
    plt.vlines(xi, yi, yp, color='orange', linestyle=':', linewidth=1.5)
plt.title(f"Iteration 5 | MSE={mse:.2f}")
plt.xlabel("Size")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)

```┌─────────────────────────────────────────────────────────┐
│                                                         │
│  MODEL:    ŷ = mx + b                                   │
│                                                         │
│  COST:     J(m,b) = (1/n) Σ (mx_i + b - y_i)²         │
│                                                         │
│  GRADIENT: ∂J/∂m  = (2/n) Σ (mx_i + b - y_i) · x_i    │
│            ∂J/∂b  = (2/n) Σ (mx_i + b - y_i) · 1       │
│                                                         │
│  UPDATE:   m ← m - α · ∂J/∂m                           │
│            b ← b - α · ∂J/∂b                           │
│                                                         │
│  REPEAT 5000 times → m and b converge to best values    │
│                                                         │
└─────────────────────────────────────────────────────────┘```

### On real Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════
# 📊 LOAD DATA
# ══════════════════════════════════════════════════════════════
df = pd.read_csv(COURSE_ROOT / "modules/06-machine-learning/datasets/external/placement.csv")

print("📋 Dataset Preview:")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Extract features
X = df['cgpa'].values  # Input: CGPA
y = df['package'].values  # Output: Placement Package

print(f"\nX (CGPA) shape: {X.shape}")
print(f"y (Package) shape: {y.shape}")

# ══════════════════════════════════════════════════════════════
# 🎓 LINEAR REGRESSION CLASS
# ══════════════════════════════════════════════════════════════

class LinearRegression:
    
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.lr = learning_rate
        self.iterations = iterations
        self.m = 0  # Slope
        self.b = 0  # Intercept
        self.loss_history = []  # Track MSE over iterations
        
    def fit(self, X, y):
        """Train the model using gradient descent"""
        n = len(X)
        
        # Initialize parameters randomly
        self.m = np.random.randn()
        self.b = np.random.randn()
        
        print(f"\n🎯 Training Started...")
        print(f"Initial: m={self.m:.4f}, b={self.b:.4f}")
        print("="*60)
        
        # Gradient Descent
        for i in range(self.iterations):
            # 1. Make predictions
            y_pred = self.m * X + self.b
            
            # 2. Calculate MSE (loss)
            mse = np.mean((y_pred - y) ** 2)
            self.loss_history.append(mse)
            
            # 3. Calculate gradients
            dm = (2/n) * np.sum((y_pred - y) * X)
            db = (2/n) * np.sum(y_pred - y)
            
            # 4. Update parameters
            self.m = self.m - self.lr * dm
            self.b = self.b - self.lr * db
            
            # Print progress every 100 iterations
            if i % 100 == 0:
                print(f"Iteration {i:4d} | MSE: {mse:8.2f} | m: {self.m:7.4f} | b: {self.b:7.4f}")
        
        print("="*60)
        print(f"✅ Training Complete!")
        print(f"Final: m={self.m:.4f}, b={self.b:.4f}")
        print(f"Final MSE: {self.loss_history[-1]:.2f}")
        
    def predict(self, X):
        """Make predictions"""
        return self.m * X + self.b
    
    def score(self, X, y):
        """Calculate R² score"""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)  # Residual sum of squares
        ss_tot = np.sum((y - np.mean(y)) ** 2)  # Total sum of squares
        r2 = 1 - (ss_res / ss_tot)
        return r2




In [ ]:
# ══════════════════════════════════════════════════════════════
# 🚀 TRAIN MODEL
# ══════════════════════════════════════════════════════════════

model = LinearRegression(learning_rate=0.01, iterations=1000)
model.fit(X, y)

# Make predictions
y_pred = model.predict(X)

# Calculate R² score
r2 = model.score(X, y)
print(f"\n📊 R² Score: {r2:.4f}")


In [ ]:

# ══════════════════════════════════════════════════════════════
# 📈 PLOT 1: ACTUAL VS PREDICTED
# ══════════════════════════════════════════════════════════════

plt.figure(figsize=(8, 5))

# Scatter actual data
plt.scatter(X, y, color='blue', s=50, alpha=0.6, label='Actual Data')

# Plot predicted line
plt.plot(X, y_pred, color='red', linewidth=2, label=f'Predicted Line (m={model.m:.2f}, b={model.b:.2f})')

plt.xlabel('CGPA', fontsize=12)
plt.ylabel('Placement Package (LPA)', fontsize=12)
plt.title('CGPA vs Placement Package', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ══════════════════════════════════════════════════════════════
# 📉 PLOT 2: LOSS (MSE) OVER ITERATIONS
# ══════════════════════════════════════════════════════════════

plt.figure(figsize=(8, 5))

plt.plot(model.loss_history, color='orange', linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('MSE (Loss)', fontsize=12)
plt.title('Loss Decrease During Training', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ══════════════════════════════════════════════════════════════
# 🎬 PLOT 3: HOW LINE UPDATES (ANIMATION STYLE)
# ══════════════════════════════════════════════════════════════

print("\n🎬 Showing how prediction line changes during training...")

# Train multiple times to capture snapshots
snapshots = [0, 10, 50, 100, 500, 999]  # Iterations to show

models_snapshots = []

# Retrain and capture models at specific iterations
temp_model = LinearRegression(learning_rate=0.01, iterations=1000)
n = len(X)

# Random initialization
temp_model.m = np.random.randn()
temp_model.b = np.random.randn()

snapshot_data = []

for i in range(1000):
    # Predict
    y_pred_temp = temp_model.m * X + temp_model.b
    
    # Calculate MSE
    mse = np.mean((y_pred_temp - y) ** 2)
    
    # Calculate gradients
    dm = (2/n) * np.sum((y_pred_temp - y) * X)
    db = (2/n) * np.sum(y_pred_temp - y)
    
    # Update
    temp_model.m = temp_model.m - temp_model.lr * dm
    temp_model.b = temp_model.b - temp_model.lr * db
    
    # Save snapshot
    if i in snapshots:
        snapshot_data.append({
            'iteration': i,
            'm': temp_model.m,
            'b': temp_model.b,
            'mse': mse
        })

# Plot all snapshots
plt.figure(figsize=(10, 6))

# Actual data
plt.scatter(X, y, color='blue', s=50, alpha=0.6, label='Actual Data', zorder=5)

# Plot each snapshot line
colors = ['red', 'orange', 'yellow', 'green', 'cyan', 'purple']

for i, snap in enumerate(snapshot_data):
    y_snap = snap['m'] * X + snap['b']
    plt.plot(X, y_snap, color=colors[i], linewidth=2, alpha=0.7,
             label=f"Iter {snap['iteration']} (MSE={snap['mse']:.1f})")

plt.xlabel('CGPA', fontsize=12)
plt.ylabel('Placement Package (LPA)', fontsize=12)
plt.title('How Prediction Line Updates During Training', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ══════════════════════════════════════════════════════════════
# 🔮 MAKE PREDICTIONS
# ══════════════════════════════════════════════════════════════

print("\n🔮 Making Predictions:")
print("="*50)

test_cgpas = [6.5, 7.0, 7.5, 8.0, 8.5, 9.0]

for cgpa in test_cgpas:
    predicted_package = model.predict(cgpa)
    print(f"CGPA: {cgpa} → Predicted Package: {predicted_package:.2f} LPA")

# ══════════════════════════════════════════════════════════════
# 📊 FINAL SUMMARY
# ══════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊 MODEL SUMMARY")
print("="*60)
print(f"Formula: Package = {model.m:.4f} × CGPA + {model.b:.4f}")
print(f"R² Score: {r2:.4f} (Higher is better, max=1.0)")
print(f"Final MSE: {model.loss_history[-1]:.2f}")
print(f"Total Iterations: {model.iterations}")
print("="*60)